# Analyse exploratoire (EDA) — Détection de fraude bancaire

**Cours MLOps — notebook 1/4.** Comprendre les transactions avant de construire le modèle.

| Notebook | Rôle |
|---|---|
| **01_eda** | comprendre les données, justifier le choix des features |
| 02_preprocessing | préparer les données, sauvegarder les processeurs utilisés par l'API |
| 03_fraud_detection_modeling | entraîner et comparer les modèles, tracer les runs dans MLflow |
| 04_mlflow_tracking | Model Registry, promotion et export du modèle de production |

**Données** : `data/raws/fraud_1M.csv` (1 296 675 transactions, 23 colonnes), versionné avec DVC.
Ce fichier correspond à `fraudTrain.csv` du jeu Kaggle *Credit Card Transactions Fraud Detection*,
produit par le simulateur **Sparkov** : les transactions sont simulées, ce qui explique certaines
observations plus bas (distance client–commerçant, préfixe des commerçants).

> **Prérequis** : `dvc pull data/raws/fraud_1M.csv` — voir `README_GITHUB_DVC_MLFLOW.md`.
> Les figures sont enregistrées dans `artifacts/eda/`.

## 0. Configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.metrics import roc_auc_score

from project_setup import ROOT, DATA_DIR, ARTIFACTS_DIR

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Chemins relatifs à la racine du dépôt : le notebook fonctionne sur toutes les machines
DATA_FILE = DATA_DIR / os.getenv('EDA_DATA_FILE', 'fraud_1M.csv')
FIG_DIR = ARTIFACTS_DIR / 'eda'
FIG_DIR.mkdir(parents=True, exist_ok=True)

LEGIT_COLOR, FRAUD_COLOR = '#2ecc71', '#e74c3c'
CLASS_COLORS = [LEGIT_COLOR, FRAUD_COLOR]
CLASS_ORDER = ['Légitime', 'Fraude']
CLASS_PALETTE = dict(zip(CLASS_ORDER, CLASS_COLORS))


def save_fig(name):
    """Enregistre la figure courante dans artifacts/eda/."""
    plt.savefig(FIG_DIR / name, dpi=150, bbox_inches='tight')


print(f"Racine du projet : {ROOT}")
print(f"Fichier analysé  : {DATA_FILE.relative_to(ROOT)}")

## 1. Chargement et aperçu

In [ ]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"{DATA_FILE} introuvable. Récupérez-le avec : dvc pull data/raws/{DATA_FILE.name}"
    )

df = pd.read_csv(DATA_FILE, parse_dates=['trans_date_trans_time', 'dob'])

print(f"Lignes   : {df.shape[0]:,}")
print(f"Colonnes : {df.shape[1]}")
print(f"Mémoire  : {df.memory_usage(deep=True).sum() / 1024**2:,.0f} Mo")
df.head()

In [ ]:
df.info()

## 2. Description des variables

La colonne **Rôle** anticipe le notebook 02 : chaque variable brute est soit conservée comme
feature, soit utilisée pour en dériver une, soit supprimée (identifiant, donnée personnelle,
redondance).

In [ ]:
features_description = [
    ('Unnamed: 0', 'Entier', "Index de la ligne dans le fichier d'origine", 'supprimée (index)'),
    ('trans_date_trans_time', 'Date-heure', 'Date et heure de la transaction', 'source : heure, jour, mois, âge'),
    ('cc_num', 'Entier', 'Numéro de carte (anonymisé)', 'supprimée (identifiant)'),
    ('merchant', 'Texte', 'Nom du commerçant', 'supprimée (identifiant, ~700 valeurs)'),
    ('category', 'Catégorielle', 'Catégorie du commerçant (grocery_pos, travel...)', 'feature catégorielle'),
    ('amt', 'Réel', 'Montant de la transaction ($)', 'feature numérique'),
    ('first', 'Texte', 'Prénom du titulaire', 'supprimée (donnée personnelle)'),
    ('last', 'Texte', 'Nom du titulaire', 'supprimée (donnée personnelle)'),
    ('gender', 'Catégorielle', 'Genre du titulaire (M/F)', 'feature catégorielle'),
    ('street', 'Texte', 'Adresse du titulaire', 'supprimée (donnée personnelle)'),
    ('city', 'Texte', 'Ville du titulaire', 'supprimée (forte cardinalité)'),
    ('state', 'Catégorielle', 'État américain (code à 2 lettres)', 'feature catégorielle'),
    ('zip', 'Entier', 'Code postal', 'feature numérique'),
    ('lat', 'Réel', 'Latitude du titulaire', 'feature numérique'),
    ('long', 'Réel', 'Longitude du titulaire', 'feature numérique'),
    ('city_pop', 'Entier', 'Population de la ville du titulaire', 'feature numérique'),
    ('job', 'Catégorielle', 'Profession du titulaire', 'feature catégorielle'),
    ('dob', 'Date', 'Date de naissance', 'source : âge'),
    ('trans_num', 'Texte', 'Identifiant unique de la transaction', 'supprimée (identifiant)'),
    ('unix_time', 'Entier', 'Horodatage Unix', 'supprimée (redondant avec la date)'),
    ('merch_lat', 'Réel', 'Latitude du commerçant', 'feature numérique'),
    ('merch_long', 'Réel', 'Longitude du commerçant', 'feature numérique'),
    ('is_fraud', 'Binaire', 'Cible : 1 = fraude, 0 = légitime', 'cible'),
]
pd.DataFrame(features_description, columns=['Variable', 'Type', 'Description', 'Rôle']).set_index('Variable')

## 3. Qualité des données

In [ ]:
quality = pd.DataFrame({
    'type': df.dtypes.astype(str),
    'valeurs_manquantes': df.isnull().sum(),
    'valeurs_distinctes': df.nunique(),
})
quality

In [ ]:
n_dup_rows = df.drop(columns=['Unnamed: 0'], errors='ignore').duplicated().sum()
n_dup_ids = df['trans_num'].duplicated().sum()
age_years = (df['trans_date_trans_time'] - df['dob']).dt.days / 365.25

print(f"Lignes dupliquées (hors index)    : {n_dup_rows:,}")
print(f"Identifiants trans_num dupliqués  : {n_dup_ids:,}")
print(f"Période couverte                  : {df['trans_date_trans_time'].min()} → {df['trans_date_trans_time'].max()}")
print(f"Montants <= 0                     : {(df['amt'] <= 0).sum():,}")
print(f"Montant médian / 99e centile      : {df['amt'].median():.2f} $ / {df['amt'].quantile(0.99):.2f} $")
print(f"Âge min / max à la transaction    : {age_years.min():.1f} / {age_years.max():.1f} ans")
print(f"Commerçants préfixés par 'fraud_' : {df['merchant'].str.startswith('fraud_').mean():.0%}")

**À retenir**

- Aucune valeur manquante ni doublon : le nettoyage se limite aux types (dates) et à la
  suppression des colonnes inutiles.
- Le préfixe `fraud_` figure sur **tous** les noms de commerçants (artefact du simulateur) :
  ce n'est pas une fuite de la cible. `merchant` est de toute façon supprimée.
- Les montants sont très asymétriques (médiane ≈ 47 $, maximum ≈ 29 000 $) : on les
  visualisera en échelle logarithmique, et on préférera un `RobustScaler` à un `StandardScaler`.

In [ ]:
print("Statistiques descriptives des variables numériques :")
df.describe().T

In [ ]:
print("Transaction au montant maximal :")
df.loc[[df['amt'].idxmax()]]

## 4. Variable cible

In [ ]:
fraud_counts = df['is_fraud'].value_counts().sort_index()
fraud_pct = df['is_fraud'].value_counts(normalize=True).sort_index() * 100

print(f"Légitimes  : {fraud_counts[0]:,} ({fraud_pct[0]:.2f} %)")
print(f"Fraudes    : {fraud_counts[1]:,} ({fraud_pct[1]:.2f} %)")
print(f"Ratio      : 1 fraude pour {fraud_counts[0] / fraud_counts[1]:.0f} transactions légitimes")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fraud_counts.plot(kind='bar', ax=axes[0], color=CLASS_COLORS)
axes[0].set_title('Nombre de transactions par classe', fontweight='bold')
axes[0].set_xticklabels(['Légitime', 'Fraude'], rotation=0)
axes[0].set_ylabel('Transactions')
axes[1].pie(fraud_counts, labels=['Légitime', 'Fraude'], autopct='%1.2f%%',
            colors=CLASS_COLORS, startangle=90, explode=(0, 0.1))
axes[1].set_title('Proportion de fraudes', fontweight='bold')
plt.tight_layout()
save_fig('fraud_distribution.png')
plt.show()

**Le piège de l'accuracy.** Un modèle qui répond toujours « légitime » obtient 99,4 %
d'accuracy tout en ne détectant aucune fraude. Avec un tel déséquilibre (≈ 1:171), on
évaluera les modèles avec la **précision**, le **rappel**, le **F1** et l'aire sous la
courbe précision–rappel (**PR-AUC**), et on traitera le déséquilibre à l'entraînement
(SMOTE ou pondération des classes, notebooks 02 et 03).

## 5. Variables dérivées pour l'analyse

- `day_period` et `distance_km` reprennent **exactement** les définitions de production
  (`backend/src/preprocessing.py`) : l'analyse justifie directement les features du modèle.
- Les tranches d'âge, de distance et de montant servent uniquement aux graphiques.

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Distance orthodromique en km (identique à backend/src/preprocessing.py)."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def get_period(hour):
    """Période de la journée, identique à la feature day_period de production."""
    if 6 <= hour < 12:
        return 'matin'
    if 12 <= hour < 18:
        return 'après-midi'
    if 18 <= hour < 22:
        return 'soirée'
    return 'nuit'


PERIOD_ORDER = ['matin', 'après-midi', 'soirée', 'nuit']
DAY_NAMES = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']
AGE_ORDER = ['<25 ans', '25-35 ans', '35-45 ans', '45-55 ans', '55-65 ans', '65+ ans']
DISTANCE_ORDER = ['<5 km', '5-20 km', '20-100 km', '>100 km']
AMOUNT_ORDER = ['<10 $', '10-50 $', '50-100 $', '100-500 $', '>500 $']

ts = df['trans_date_trans_time']
df['classe'] = df['is_fraud'].map({0: 'Légitime', 1: 'Fraude'})   # libellé pour les graphiques
df['hour'] = ts.dt.hour
df['day_of_week'] = ts.dt.dayofweek
df['month'] = ts.dt.month
df['year_month'] = ts.dt.to_period('M').astype(str)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['day_period'] = df['hour'].apply(get_period)
df['age'] = (ts - df['dob']).dt.days / 365.25
df['distance_km'] = haversine_distance(df['lat'], df['long'], df['merch_lat'], df['merch_long'])

df['age_category'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 65, np.inf], labels=AGE_ORDER, right=False)
df['distance_category'] = pd.cut(df['distance_km'], bins=[0, 5, 20, 100, np.inf], labels=DISTANCE_ORDER, right=False)
df['amount_category'] = pd.cut(df['amt'], bins=[0, 10, 50, 100, 500, np.inf], labels=AMOUNT_ORDER, right=False)

new_columns = ['hour', 'day_of_week', 'month', 'year_month', 'is_weekend', 'day_period',
               'age', 'age_category', 'distance_km', 'distance_category', 'amount_category']
df[new_columns].head()

In [ ]:
def fraud_rate_by(column, order=None, min_count=0):
    """Nombre de fraudes, de transactions et taux de fraude (%) par modalité."""
    stats = df.groupby(column, observed=True)['is_fraud'].agg(fraudes='sum', transactions='count')
    stats['taux_fraude_%'] = stats['fraudes'] / stats['transactions'] * 100
    if order is not None:
        stats = stats.reindex(order)
    if min_count:
        stats = stats[stats['transactions'] >= min_count]
    return stats

## 6. Montants des transactions

In [ ]:
df.groupby('is_fraud')['amt'].describe().rename(index={0: 'Légitime', 1: 'Fraude'})

In [ ]:
amount_stats = fraud_rate_by('amount_category', AMOUNT_ORDER)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

sns.histplot(data=df, x='amt', hue='classe', hue_order=CLASS_ORDER, palette=CLASS_PALETTE,
             bins=60, log_scale=True, stat='density', common_norm=False, ax=axes[0, 0])
axes[0, 0].set_title('Distribution des montants (échelle log, densité par classe)', fontweight='bold')

sns.boxplot(data=df, x='classe', y='amt', hue='classe', order=CLASS_ORDER, palette=CLASS_PALETTE,
            legend=False, showfliers=False, ax=axes[0, 1])
axes[0, 1].set_yscale('log')
axes[0, 1].set_title('Montants par classe (sans valeurs extrêmes)', fontweight='bold')

amount_stats['taux_fraude_%'].plot(kind='bar', ax=axes[1, 0], color=FRAUD_COLOR)
axes[1, 0].set_title('Taux de fraude par tranche de montant', fontweight='bold')
axes[1, 0].set_ylabel('Taux de fraude (%)')
axes[1, 0].tick_params(axis='x', rotation=0)

sns.ecdfplot(data=df, x='amt', hue='classe', hue_order=CLASS_ORDER, palette=CLASS_PALETTE,
             log_scale=True, ax=axes[1, 1])
axes[1, 1].set_title('Fonction de répartition des montants', fontweight='bold')

plt.tight_layout()
save_fig('amount_analysis.png')
plt.show()
amount_stats

**À retenir.** Les fraudes se concentrent sur les montants élevés (moyenne ≈ 530 $ contre
≈ 68 $ pour les transactions légitimes), mais **plafonnent autour de 1 400 $** alors que des
transactions légitimes dépassent 28 000 $. `amt` est la variable la plus discriminante du jeu
de données (confirmé en section 11).

## 7. Analyse temporelle

In [ ]:
hourly = fraud_rate_by('hour')
daily = fraud_rate_by('day_of_week').rename(index=dict(enumerate(DAY_NAMES)))
period = fraud_rate_by('day_period', PERIOD_ORDER)

print("Heures les plus risquées :")
print(hourly.sort_values('taux_fraude_%', ascending=False).head(6).round(3))

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

axes[0, 0].plot(hourly.index, hourly['taux_fraude_%'], marker='o', color=FRAUD_COLOR, linewidth=2)
axes[0, 0].fill_between(hourly.index, hourly['taux_fraude_%'], alpha=0.3, color=FRAUD_COLOR)
axes[0, 0].set_title('Taux de fraude par heure', fontweight='bold')
axes[0, 0].set_xlabel('Heure')
axes[0, 0].set_ylabel('Taux de fraude (%)')
axes[0, 0].set_xticks(range(0, 24, 2))

hour_profile = pd.crosstab(df['hour'], df['is_fraud'], normalize='columns') * 100
hour_profile.plot(kind='bar', ax=axes[0, 1], color=CLASS_COLORS, width=0.8)
axes[0, 1].set_title('Répartition horaire de chaque classe (% des transactions de la classe)', fontweight='bold')
axes[0, 1].set_ylabel('%')
axes[0, 1].legend(['Légitime', 'Fraude'])

daily['taux_fraude_%'].plot(kind='bar', ax=axes[1, 0], color=FRAUD_COLOR)
axes[1, 0].set_title('Taux de fraude par jour de la semaine', fontweight='bold')
axes[1, 0].set_ylabel('Taux de fraude (%)')
axes[1, 0].tick_params(axis='x', rotation=45)

period['taux_fraude_%'].plot(kind='bar', ax=axes[1, 1], color=FRAUD_COLOR)
axes[1, 1].set_title('Taux de fraude par période (feature day_period)', fontweight='bold')
axes[1, 1].set_ylabel('Taux de fraude (%)')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
save_fig('temporal_analysis.png')
plt.show()
period

**À retenir.** La fraude est nocturne : le taux culmine à 22 h–23 h puis reste élevé jusqu'à
3 h. C'est ce qui justifie la période `nuit` = **22 h–6 h** de la feature `day_period`. Le jour de
la semaine joue peu.

La relation entre l'heure et la fraude n'est **pas monotone** (pics aux deux extrémités de
l'échelle 0–23) : un modèle linéaire exploiterait mal la variable `trans_hour` brute, alors que
les arbres de décision et la feature catégorielle `day_period` la captent bien.

In [ ]:
monthly = df.groupby('year_month')['is_fraud'].agg(transactions='count', taux='mean')
monthly['taux'] *= 100

fig, ax1 = plt.subplots(figsize=(16, 5))
ax1.bar(monthly.index, monthly['transactions'], color='#3498db', alpha=0.6)
ax1.set_ylabel('Transactions', color='#3498db')
ax1.tick_params(axis='x', rotation=45)
ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly['taux'], color=FRAUD_COLOR, marker='o', linewidth=2)
ax2.set_ylabel('Taux de fraude (%)', color=FRAUD_COLOR)
ax2.grid(False)
ax1.set_title('Évolution mensuelle du volume et du taux de fraude', fontweight='bold')
plt.tight_layout()
save_fig('monthly_evolution.png')
plt.show()

**Pourquoi c'est important en MLOps.** Le volume et le taux de fraude varient d'un mois à
l'autre (saisonnalité, pic de décembre). Un modèle entraîné sur une période n'a aucune garantie
sur la suivante : c'est la motivation de la **détection de drift** (Evidently, pipeline Jenkins)
et du ré-entraînement. Cela invite aussi à valider le modèle sur une **période postérieure**
à l'entraînement, en complément du découpage aléatoire.

## 8. Catégories de commerçants

In [ ]:
category_stats = fraud_rate_by('category').sort_values('taux_fraude_%', ascending=False)
category_stats

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

category_stats['taux_fraude_%'].sort_values().plot(kind='barh', ax=axes[0, 0], color=FRAUD_COLOR)
axes[0, 0].set_title('Taux de fraude par catégorie', fontweight='bold')
axes[0, 0].set_xlabel('Taux de fraude (%)')

category_stats['transactions'].sort_values().plot(kind='barh', ax=axes[0, 1], color='#3498db')
axes[0, 1].set_title('Volume de transactions par catégorie', fontweight='bold')

category_stats['fraudes'].sort_values().plot(kind='barh', ax=axes[1, 0], color=FRAUD_COLOR)
axes[1, 0].set_title('Nombre de fraudes par catégorie', fontweight='bold')

category_amount = df.groupby(['category', 'is_fraud'])['amt'].mean().unstack(fill_value=0)
category_amount.plot(kind='barh', ax=axes[1, 1], color=CLASS_COLORS)
axes[1, 1].set_title('Montant moyen par catégorie et par classe', fontweight='bold')
axes[1, 1].legend(['Légitime', 'Fraude'])

plt.tight_layout()
save_fig('category_analysis.png')
plt.show()

**À retenir.** Les catégories en ligne (`shopping_net`, `misc_net`) et `grocery_pos` cumulent
les taux de fraude les plus élevés, avec des montants frauduleux nettement supérieurs aux
montants légitimes dans ces catégories. `category` est une feature importante, et son
interaction avec le montant est exploitable par les modèles d'arbres.

## 9. Analyse géographique

In [ ]:
MIN_TRANSACTIONS = 1_000   # effectif minimal pour qu'un taux soit interprétable

state_all = fraud_rate_by('state').sort_values('taux_fraude_%', ascending=False)
state_reliable = fraud_rate_by('state', min_count=MIN_TRANSACTIONS).sort_values('taux_fraude_%', ascending=False)

print("Top 5 sans filtre d'effectif :")
print(state_all.head(5).round(3))
print(f"\nTop 10 parmi les États avec au moins {MIN_TRANSACTIONS:,} transactions :")
print(state_reliable.head(10).round(3))

In [ ]:
distance_by_class = df.groupby('is_fraud')['distance_km'].describe().rename(index={0: 'Légitime', 1: 'Fraude'})
distance_stats = fraud_rate_by('distance_category', DISTANCE_ORDER)
city_pop_by_class = df.groupby('is_fraud')['city_pop'].agg(['mean', 'median']).rename(index={0: 'Légitime', 1: 'Fraude'})

print("Distance client-commerçant (km) :")
print(distance_by_class.round(2))
print("\nTaux de fraude par tranche de distance :")
print(distance_stats.round(3))
print("\nPopulation de la ville du titulaire — moyenne et médiane :")
print(city_pop_by_class.round(0))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

state_reliable.head(15)['taux_fraude_%'].sort_values().plot(kind='barh', ax=axes[0, 0], color=FRAUD_COLOR)
axes[0, 0].set_title(f'Top 15 États (≥ {MIN_TRANSACTIONS:,} transactions)', fontweight='bold')
axes[0, 0].set_xlabel('Taux de fraude (%)')

sns.boxplot(data=df, x='classe', y='distance_km', hue='classe', order=CLASS_ORDER,
            palette=CLASS_PALETTE, legend=False, ax=axes[0, 1])
axes[0, 1].set_title('Distance client-commerçant par classe', fontweight='bold')

distance_stats['taux_fraude_%'].plot(kind='bar', ax=axes[1, 0], color=FRAUD_COLOR)
axes[1, 0].set_title('Taux de fraude par tranche de distance', fontweight='bold')
axes[1, 0].set_ylabel('Taux de fraude (%)')
axes[1, 0].tick_params(axis='x', rotation=0)

sns.boxplot(data=df, x='classe', y='city_pop', hue='classe', order=CLASS_ORDER,
            palette=CLASS_PALETTE, legend=False, ax=axes[1, 1])
axes[1, 1].set_yscale('log')
axes[1, 1].set_title('Population de la ville (échelle log)', fontweight='bold')

plt.tight_layout()
save_fig('geographic_analysis.png')
plt.show()

**À retenir**

- **Petits effectifs.** Un État peut afficher 100 % de fraude sur une poignée de transactions
  (9 sur 9 pour `DE` dans le jeu complet) : ce taux n'a aucune valeur statistique. On ne
  compare que des groupes d'effectif suffisant.
- **Distance.** Les distributions sont pratiquement identiques pour les deux classes
  (≈ 76 km en moyenne) : dans ce jeu simulé, la position du commerçant est tirée au hasard
  autour du client. `distance_km` reste une feature, mais **il ne faut pas en attendre de
  pouvoir prédictif** — contrairement à ce que suggérerait l'intuition métier.
- **Population.** La moyenne est tirée par quelques très grandes villes ; c'est la médiane et
  le boxplot en échelle log qu'il faut comparer. L'écart entre classes est faible.

## 10. Analyse démographique

In [ ]:
gender_stats = fraud_rate_by('gender')
age_stats = fraud_rate_by('age_category', AGE_ORDER)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

gender_stats['taux_fraude_%'].plot(kind='bar', ax=axes[0], color=FRAUD_COLOR)
axes[0].set_title('Taux de fraude par genre', fontweight='bold')
axes[0].set_ylabel('Taux de fraude (%)')
axes[0].tick_params(axis='x', rotation=0)

sns.kdeplot(data=df, x='age', hue='classe', hue_order=CLASS_ORDER, palette=CLASS_PALETTE,
            common_norm=False, ax=axes[1])
axes[1].set_title("Distribution de l'âge par classe", fontweight='bold')

age_stats['taux_fraude_%'].plot(kind='bar', ax=axes[2], color=FRAUD_COLOR)
axes[2].set_title("Taux de fraude par tranche d'âge", fontweight='bold')
axes[2].set_ylabel('Taux de fraude (%)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
save_fig('demographic_analysis.png')
plt.show()
pd.concat({'genre': gender_stats, 'âge': age_stats})

**À retenir.** Les clients de plus de 55 ans et de moins de 25 ans sont un peu plus exposés ;
l'écart entre genres est faible. `age` apporte une information modeste mais réelle.

## 11. Pouvoir discriminant des variables

Deux mesures simples, adaptées à une cible binaire déséquilibrée :

- **AUC univariée** (variables numériques) : on utilise la variable seule comme score.
  0,5 = aucune information, 1 = séparation parfaite. Elle ne capte que les relations
  **monotones**.
- **V de Cramér** (variables catégorielles) : intensité de l'association avec la cible,
  entre 0 et 1.

In [ ]:
numeric_candidates = ['amt', 'age', 'hour', 'city_pop', 'distance_km', 'lat', 'long',
                      'merch_lat', 'merch_long', 'zip', 'month', 'day_of_week']
auc_rows = []
for col in numeric_candidates:
    auc = roc_auc_score(df['is_fraud'], df[col])
    auc_rows.append({'variable': col, 'auc_univariee': max(auc, 1 - auc)})   # 0,3 vaut 0,7 en sens inverse
auc_table = pd.DataFrame(auc_rows).sort_values('auc_univariee', ascending=False)


def cramers_v(x, y):
    table = pd.crosstab(x, y)
    chi2 = chi2_contingency(table, correction=False)[0]
    return np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))


categorical_candidates = ['category', 'day_period', 'amount_category', 'age_category',
                          'state', 'job', 'gender', 'is_weekend']
cramer_table = pd.DataFrame(
    [{'variable': col, 'v_de_cramer': cramers_v(df[col], df['is_fraud'])} for col in categorical_candidates]
).sort_values('v_de_cramer', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
auc_table.set_index('variable')['auc_univariee'].sort_values().plot(kind='barh', ax=axes[0], color='#3498db')
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=1)
axes[0].set_xlim(0.45, 1)
axes[0].set_title('AUC univariée (variables numériques)', fontweight='bold')
cramer_table.set_index('variable')['v_de_cramer'].sort_values().plot(kind='barh', ax=axes[1], color=FRAUD_COLOR)
axes[1].set_title('V de Cramér (variables catégorielles)', fontweight='bold')
plt.tight_layout()
save_fig('discriminant_power.png')
plt.show()

pd.concat([auc_table.reset_index(drop=True), cramer_table.reset_index(drop=True)], axis=1)

**À retenir.** `amt` domine nettement. `hour` obtient une AUC univariée faible alors que la
section 7 montre un effet fort : l'AUC ne mesure que les relations **monotones**. Le V de
Cramér de `day_period`, qui découpe l'heure en périodes, rend compte de cet effet. Les
coordonnées, `zip` et `distance_km` sont proches de 0,5 : peu ou pas d'information
individuellement.

## 12. Corrélations

On utilise la corrélation de **Spearman** (sur les rangs) : robuste aux montants extrêmes et
sensible à toute relation monotone, pas seulement linéaire.

In [ ]:
corr_cols = ['amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long',
             'hour', 'day_of_week', 'month', 'age', 'distance_km', 'is_fraud']
correlation_matrix = df[corr_cols].corr(method='spearman')

plt.figure(figsize=(13, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Matrice de corrélation de Spearman', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
save_fig('correlation_matrix.png')
plt.show()

correlation_matrix['is_fraud'].drop('is_fraud').sort_values(key=abs, ascending=False)

**À retenir.** Les coordonnées du commerçant sont quasi identiques à celles du client
(corrélation ≈ 1) : ces variables sont **redondantes**. Ce n'est pas gênant pour des arbres
de décision, mais cela le serait pour une régression linéaire (colinéarité). Aucune
corrélation forte avec la cible hormis `amt` : la fraude se détecte par **combinaisons** de
variables, d'où l'intérêt de modèles non linéaires.

## 13. Patterns de fraude

In [ ]:
card_fraud_counts = df[df['is_fraud'] == 1].groupby('cc_num').size()
multi_fraud_cards = card_fraud_counts[card_fraud_counts > 1]
print(f"Cartes ayant subi au moins une fraude   : {len(card_fraud_counts):,}")
print(f"... dont plusieurs fraudes               : {len(multi_fraud_cards):,}")
print(f"Fraudes par carte fraudée (moyenne / max) : {card_fraud_counts.mean():.1f} / {card_fraud_counts.max()}")

weekend_stats = fraud_rate_by('is_weekend').rename(index={0: 'Semaine', 1: 'Week-end'})
merchant_stats = fraud_rate_by('merchant', min_count=MIN_TRANSACTIONS).sort_values('taux_fraude_%', ascending=False)
cross_analysis = pd.crosstab(df['category'], df['day_period'], values=df['is_fraud'], aggfunc='mean').reindex(columns=PERIOD_ORDER) * 100
weekend_stats

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

card_fraud_counts.value_counts().sort_index().plot(kind='bar', ax=axes[0, 0], color=FRAUD_COLOR)
axes[0, 0].set_title('Nombre de fraudes par carte fraudée', fontweight='bold')
axes[0, 0].set_xlabel('Fraudes sur la même carte')
axes[0, 0].set_ylabel('Cartes')

weekend_stats['taux_fraude_%'].plot(kind='bar', ax=axes[0, 1], color=FRAUD_COLOR)
axes[0, 1].set_title('Taux de fraude : semaine vs week-end', fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=0)

sns.heatmap(cross_analysis, annot=True, fmt='.2f', cmap='Reds', ax=axes[1, 0],
            cbar_kws={'label': 'Taux de fraude (%)'})
axes[1, 0].set_title('Taux de fraude : catégorie × période', fontweight='bold')

merchant_stats.head(10)['taux_fraude_%'].sort_values().plot(kind='barh', ax=axes[1, 1], color=FRAUD_COLOR)
axes[1, 1].set_title(f'Top 10 commerçants par taux de fraude (≥ {MIN_TRANSACTIONS:,} transactions)', fontweight='bold')

plt.tight_layout()
save_fig('fraud_patterns.png')
plt.show()

**À retenir.** Une carte fraudée subit en moyenne une dizaine de fraudes : les fraudes
arrivent **en rafales**. Des features de *vélocité* (nombre de transactions et montant cumulé
de la carte sur les dernières 24 h) seraient très prédictives. Elles exigent toutefois un
historique par carte au moment de l'inférence (base de features en ligne) : c'est une piste
d'évolution de l'architecture, hors du périmètre actuel de l'API.

La croisée catégorie × période montre que le risque est porté par certaines combinaisons
(achats en ligne la nuit) plus que par chaque variable isolément.

## 14. Synthèse

In [ ]:
top_hour = hourly['taux_fraude_%'].idxmax()
print("SYNTHÈSE CHIFFRÉE")
print("=" * 70)
print(f"Transactions            : {len(df):,} ({df['trans_date_trans_time'].min():%Y-%m-%d} → {df['trans_date_trans_time'].max():%Y-%m-%d})")
print(f"Taux de fraude          : {df['is_fraud'].mean():.2%} (1 pour {fraud_counts[0] / fraud_counts[1]:.0f})")
print(f"Montant moyen           : légitime {df.loc[df.is_fraud == 0, 'amt'].mean():.0f} $ | fraude {df.loc[df.is_fraud == 1, 'amt'].mean():.0f} $")
print(f"Pertes totales (fraudes): {df.loc[df.is_fraud == 1, 'amt'].sum():,.0f} $")
print(f"Heure la plus risquée   : {top_hour} h ({hourly.loc[top_hour, 'taux_fraude_%']:.2f} %)")
print(f"Catégories à risque     : {', '.join(category_stats.index[:3])}")
print(f"Variable la plus discriminante : {auc_table.iloc[0]['variable']} (AUC univariée {auc_table.iloc[0]['auc_univariee']:.3f})")
print(f"Figures enregistrées    : {', '.join(sorted(p.name for p in FIG_DIR.glob('*.png')))}")

### Recommandations pour la suite

| Constat | Conséquence pour le pipeline |
|---|---|
| Déséquilibre ≈ 1:171 | SMOTE ou pondération **sur le train uniquement** ; évaluer avec précision, rappel, F1, PR-AUC — jamais l'accuracy seule |
| `amt`, période (nuit 22 h–6 h), `category`, `age` porteurs de signal | features conservées (`amt_category`, `day_period` dérivées en production) |
| `distance_km`, coordonnées, `zip`, `city_pop` peu informatives | conservées (sans danger pour des arbres), mais ne pas en attendre d'effet |
| Taux par État instables sur petits effectifs | ne pas surinterpréter `state` ; seuil d'effectif dans les analyses |
| Identifiants et données personnelles | supprimés (`cc_num`, `trans_num`, `first`, `last`, `street`...) |
| Fraudes groupées par carte | piste : features de vélocité (nécessite un historique en ligne) |
| Volume et taux variables dans le temps | surveiller le drift en production, valider aussi sur une période postérieure |

**Étape suivante** : `02_preprocessing.ipynb`. Après une ré-exécution, versionner les figures avec
`dvc add artifacts` puis `dvc push` (voir le README).